# Part 1 — Sorting Algorithm Comparison
**Algorithm Analysis and Simulation Toolkit | Term 2, SY 2025–2026**

---
## Algorithm Explanations

| Algorithm | Strategy | Best | Average | Worst | Space |
|---|---|---|---|---|---|
| Bubble Sort | Swap adjacent out-of-order pairs | O(n) | O(n²) | O(n²) | O(1) |
| Selection Sort | Find min, place at front each pass | O(n²) | O(n²) | O(n²) | O(1) |
| Insertion Sort | Insert each element into sorted prefix | O(n) | O(n²) | O(n²) | O(1) |
| Merge Sort | Divide, sort recursively, merge | O(n log n) | O(n log n) | O(n log n) | O(n) |
| Quick Sort | Partition around pivot | O(n log n) | O(n log n) | O(n²) | O(log n) |
| Random-Quick Sort | Quick Sort with random pivot | O(n log n) | O(n log n) | O(n²) rare | O(log n) |
| Counting Sort | Count frequencies, rebuild | O(n+k) | O(n+k) | O(n+k) | O(k) |
| Radix Sort | Sort digit by digit (LSD) | O(nk) | O(nk) | O(nk) | O(n+k) |

---
## Algorithm Flowcharts

### Bubble Sort
```
START → Set i=0
  └─► i < n? ──NO──► END
        │YES
        └─► Set j=0, swapped=False
              └─► j < n-i-1? ──NO──► swapped? ──YES──► i++, loop
                        │YES                  └──NO──► EARLY EXIT
                        └─► arr[j] > arr[j+1]?
                              │YES                 │NO
                              └─► swap; swapped=T  └─► j++
                              └─► j++  (loop back)
```
### Merge Sort
```
merge_sort(arr)
  └─► len ≤ 1? → return arr
  └─► mid = len//2
  └─► left  = merge_sort(arr[:mid])   ← recursive
  └─► right = merge_sort(arr[mid:])   ← recursive
  └─► merge(left, right) → compare heads → append smaller → return
```
### Quick Sort
```
quick_sort(lo, hi)
  └─► lo < hi? → return
  └─► pivot = arr[hi]  (or random for Random-Quick)
  └─► partition: elements ≤ pivot → left side, > pivot → right
  └─► quick_sort(lo, p-1) ← recursive
  └─► quick_sort(p+1, hi) ← recursive
```

---
## Program Instructions
1. Run **Cell 2** first to load all 8 algorithms.
2. **Benchmark**: choose size (default 1000), input type, algorithms → click Run.
3. **Trace**: enter a small array → watch each swap or see before/after.
4. **Case Analysis**: compare best / average / worst times side by side.
5. **Test Cases**: automatically verify all algorithms are correct.


In [1]:
import time, random
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── numpy is used for fast dataset generation and correctness checks ──────────
# The sorting algorithms themselves are pure Python so comparisons/swaps
# can be counted accurately — that is a spec requirement.

# ── BUBBLE SORT ───────────────────────────────────────────────────────────────
# Repeatedly swaps adjacent elements if out of order.
# Early-exit: stops if a full pass produces zero swaps (already sorted).
def bubble_sort(arr):
    a = arr.copy(); n = len(a); comps = swaps = 0
    for i in range(n):
        swapped = False
        for j in range(0, n - i - 1):
            comps += 1
            if a[j] > a[j + 1]:
                a[j], a[j+1] = a[j+1], a[j]; swaps += 1; swapped = True
        if not swapped: break
    return a, comps, swaps

# ── SELECTION SORT ────────────────────────────────────────────────────────────
# Each pass finds the minimum in the unsorted portion and places it at the front.
def selection_sort(arr):
    a = arr.copy(); n = len(a); comps = swaps = 0
    for i in range(n):
        mi = i
        for j in range(i + 1, n):
            comps += 1
            if a[j] < a[mi]: mi = j
        if mi != i: a[i], a[mi] = a[mi], a[i]; swaps += 1
    return a, comps, swaps

# ── INSERTION SORT ────────────────────────────────────────────────────────────
# Builds a sorted prefix; inserts each new element at its correct position.
def insertion_sort(arr):
    a = arr.copy(); comps = swaps = 0
    for i in range(1, len(a)):
        key = a[i]; j = i - 1
        while j >= 0:
            comps += 1
            if a[j] > key: a[j+1] = a[j]; swaps += 1; j -= 1
            else: break
        a[j+1] = key
    return a, comps, swaps

# ── MERGE SORT ────────────────────────────────────────────────────────────────
# Divide-and-conquer: split, sort recursively, merge. Stable O(n log n).
def merge_sort(arr):
    comps = [0]
    def _merge(L, R):
        out = []; i = j = 0
        while i < len(L) and j < len(R):
            comps[0] += 1
            if L[i] <= R[j]: out.append(L[i]); i += 1
            else: out.append(R[j]); j += 1
        out.extend(L[i:]); out.extend(R[j:]); return out
    def _sort(a):
        if len(a) <= 1: return a
        m = len(a) // 2
        return _merge(_sort(a[:m]), _sort(a[m:]))
    return _sort(list(arr)), comps[0], 0

# ── QUICK SORT (deterministic — last element pivot) ───────────────────────────
# Partitions around a pivot. Worst case O(n²) on sorted input.
def quick_sort(arr):
    comps = [0]; swaps = [0]
    def _part(a, lo, hi):
        pv = a[hi]; i = lo - 1
        for j in range(lo, hi):
            comps[0] += 1
            if a[j] <= pv: i += 1; a[i], a[j] = a[j], a[i]; swaps[0] += 1
        a[i+1], a[hi] = a[hi], a[i+1]; swaps[0] += 1; return i + 1
    def _sort(a, lo, hi):
        if lo < hi: p = _part(a, lo, hi); _sort(a, lo, p-1); _sort(a, p+1, hi)
    a = list(arr); _sort(a, 0, len(a)-1); return a, comps[0], swaps[0]

# ── RANDOMIZED QUICK SORT ─────────────────────────────────────────────────────
# Random pivot choice eliminates worst-case O(n²) on sorted input.
def random_quick_sort(arr):
    comps = [0]; swaps = [0]
    def _part(a, lo, hi):
        r = random.randint(lo, hi)
        a[r], a[hi] = a[hi], a[r]; swaps[0] += 1
        pv = a[hi]; i = lo - 1
        for j in range(lo, hi):
            comps[0] += 1
            if a[j] <= pv: i += 1; a[i], a[j] = a[j], a[i]; swaps[0] += 1
        a[i+1], a[hi] = a[hi], a[i+1]; swaps[0] += 1; return i + 1
    def _sort(a, lo, hi):
        if lo < hi: p = _part(a, lo, hi); _sort(a, lo, p-1); _sort(a, p+1, hi)
    a = list(arr); _sort(a, 0, len(a)-1); return a, comps[0], swaps[0]

# ── COUNTING SORT ─────────────────────────────────────────────────────────────
# Non-comparison sort for non-negative integers. Uses numpy for the count
# array allocation — the counting loop itself stays Python for metric tracking.
def counting_sort(arr):
    if len(arr) == 0: return [], 0, 0
    a = list(arr); mx = int(np.max(arr))
    cnt = np.zeros(mx + 1, dtype=np.int64); comps = 0
    for v in a: cnt[v] += 1; comps += 1
    out = []
    for i, c in enumerate(cnt): out.extend([int(i)] * int(c))
    return out, comps, 0

# ── RADIX SORT (LSD, base-10) ─────────────────────────────────────────────────
# Sorts digit by digit from least to most significant. O(nk), no comparisons.
def radix_sort(arr):
    if len(arr) == 0: return [], 0, 0
    a = list(arr); comps = 0; exp = 1; mx = int(np.max(arr))
    while mx // exp > 0:
        buckets = [[] for _ in range(10)]
        for num in a: buckets[(num // exp) % 10].append(num); comps += 1
        a = [x for b in buckets for x in b]
        exp *= 10
    return a, comps, 0

# ── ALGORITHM REGISTRY ────────────────────────────────────────────────────────
ALGORITHMS = {
    'Bubble Sort':       bubble_sort,
    'Selection Sort':    selection_sort,
    'Insertion Sort':    insertion_sort,
    'Merge Sort':        merge_sort,
    'Quick Sort':        quick_sort,
    'Random-Quick Sort': random_quick_sort,
    'Counting Sort':     counting_sort,
    'Radix Sort':        radix_sort,
}

# ── FAST DATASET GENERATOR (numpy) ───────────────────────────────────────────
# np.random.randint is ~10x faster than a Python list comprehension.
# Returns a plain Python list so all algorithms work unchanged.
def make_dataset(size, itype='Random'):
    if itype == 'Random':
        return np.random.randint(0, 10000, size=size).tolist()
    elif itype == 'Sorted (best case)':
        return list(range(size))
    else:
        return list(range(size, 0, -1))

print('=== Part 1: Sorting Algorithm Comparison ===')
print(f'  {len(ALGORITHMS)} algorithms loaded  |  numpy-accelerated dataset generation enabled')
print('  Scroll down to run the benchmark, trace, case analysis, and tests.')


=== Part 1: Sorting Algorithm Comparison ===
  8 algorithms loaded  |  numpy-accelerated dataset generation enabled
  Scroll down to run the benchmark, trace, case analysis, and tests.


## Benchmark
Default dataset size: **1000**. numpy generates datasets instantly — no lag at n=5000.

In [2]:
# ── BENCHMARK ─────────────────────────────────────────────────────────────────
# Generates a dataset with numpy (fast even at n=5000), then runs every
# selected algorithm once and records time, comparisons, and swaps.
# Displays: input preview, sorted output preview, and the performance table.

w_size = widgets.Dropdown(
    options=[100, 500, 1000, 2000, 5000], value=1000,
    description='Dataset Size:', style={'description_width': 'initial'}
)
w_itype = widgets.RadioButtons(
    options=['Random', 'Sorted (best case)', 'Reversed (worst case)'],
    value='Random', description='Input type:', style={'description_width': 'initial'}
)
w_algos = widgets.SelectMultiple(
    options=list(ALGORITHMS.keys()), value=list(ALGORITHMS.keys()),
    description='Algorithms:', rows=8, style={'description_width': 'initial'}
)
w_show_out = widgets.Checkbox(value=True, description='Show sorted output preview')
w_run = widgets.Button(
    description='▶  Run Benchmark',
    button_style='success', layout=widgets.Layout(width='170px', height='36px')
)
out_b = widgets.Output()

def run_benchmark(_):
    with out_b:
        clear_output(wait=True)
        size  = w_size.value
        algos = list(w_algos.value)
        itype = w_itype.value

        if not algos:
            print('⚠  Select at least one algorithm.'); return

        # numpy-generated dataset — instant even at n=5000
        dataset = make_dataset(size, itype)
        print(f'Input   : {dataset[:10]}...  (n={size}, type={itype})')
        print()

        results = []
        for name in algos:
            t0 = time.perf_counter()
            sorted_out, c, s = ALGORITHMS[name](dataset)
            elapsed = round((time.perf_counter() - t0) * 1000, 2)
            results.append({
                'Algorithm':   name,
                'Time (ms)':   elapsed,
                'Comparisons': c,
                'Swaps':       s,
                'out':         sorted_out
            })
        results.sort(key=lambda x: x['Time (ms)'])

        # sorted output — spec requirement
        if w_show_out.value:
            print(f'Sorted  : {results[0]["out"][:20]}...  (first 20 of {size})')
            print()

        # performance table — exact spec format
        print(f'Dataset Size: {size}')
        print()
        W = [22, 12, 16, 10]
        hdr = f"{'Algorithm':<{W[0]}} {'Time (ms)':>{W[1]}} {'Comparisons':>{W[2]}} {'Swaps':>{W[3]}}"
        print(hdr)
        print('─' * len(hdr))
        for r in results:
            print(f"{r['Algorithm']:<{W[0]}} {r['Time (ms)']:>{W[1]}.2f} "
                  f"{r['Comparisons']:>{W[2]},} {r['Swaps']:>{W[3]},}")
        print()
        fastest = results[0]; slowest = results[-1]
        print(f'Fastest : {fastest["Algorithm"]}  ({fastest["Time (ms)"]} ms)')
        print(f'Slowest : {slowest["Algorithm"]}  ({slowest["Time (ms)"]} ms)')
        if fastest['Time (ms)'] > 0:
            print(f'Ratio   : {round(slowest["Time (ms)"] / fastest["Time (ms)"], 1)}x slower')

w_run.on_click(run_benchmark)
display(widgets.VBox([
    widgets.HBox([w_size, w_itype]),
    w_algos, w_show_out, w_run, out_b
]))


## Step-by-Step Trace

In [7]:
# ── STEP-BY-STEP TRACE ────────────────────────────────────────────────────────
# Bubble Sort: shows every individual swap with the array state after each step.
# Other algorithms: shows input, sorted output, comparisons, and swaps.
# Array capped at 20 elements for readability.

w_tarr = widgets.Text(
    value='9, 3, 7, 1, 5, 8, 2, 6, 4',
    description='Array (csv):', layout=widgets.Layout(width='370px'),
    style={'description_width': 'initial'}
)
w_talgo = widgets.Dropdown(
    options=list(ALGORITHMS.keys()), value='Bubble Sort',
    description='Algorithm:', style={'description_width': 'initial'}
)
w_trun = widgets.Button(
    description='▶  Show Trace',
    button_style='primary', layout=widgets.Layout(width='140px', height='36px')
)
out_t = widgets.Output()

def show_trace(_):
    with out_t:
        clear_output(wait=True)
        try:
            arr = [int(x.strip()) for x in w_tarr.value.split(',') if x.strip()]
        except ValueError:
            print('⚠  Use comma-separated integers (e.g. 5, 3, 8, 1)'); return
        if len(arr) < 2: print('⚠  Need at least 2 elements.'); return
        if len(arr) > 20: arr = arr[:20]; print('⚠  Capped at 20 elements.\n')

        name = w_talgo.value
        print(f'=== {name} — Step-by-Step Trace ===')
        print(f'Input  : {arr}\n')

        if name == 'Bubble Sort':
            a = arr[:]; n = len(a); step = 0
            for i in range(n):
                pass_swap = False
                for j in range(0, n - i - 1):
                    if a[j] > a[j+1]:
                        a[j], a[j+1] = a[j+1], a[j]; step += 1; pass_swap = True
                        print(f'  Step {step:>3} | swap {a[j+1]} ↔ {a[j]} at [{j},{j+1}] → {a}')
                if not pass_swap:
                    print(f'  Pass {i+1}: no swaps — early exit.'); break
            print(f'\nSorted : {a}')
            print(f'Total swaps: {step}')
        else:
            s, comps, swaps = ALGORITHMS[name](arr)
            print(f'Sorted      : {s}')
            print(f'Comparisons : {comps}')
            print(f'Swaps       : {swaps}')

w_trun.on_click(show_trace)
display(widgets.VBox([widgets.HBox([w_talgo, w_tarr]), w_trun, out_t]))


## Best / Average / Worst Case Analysis

In [8]:
# ── BEST / AVERAGE / WORST CASE ANALYSIS ─────────────────────────────────────
# Uses numpy to generate datasets instantly. Compares all algorithms across
# sorted, random, and reversed inputs of the same size.

w_cn = widgets.IntSlider(
    value=500, min=50, max=2000, step=50,
    description='n:', style={'description_width': 'initial'}
)
w_crun = widgets.Button(
    description='▶  Run Case Analysis',
    button_style='warning', layout=widgets.Layout(width='190px', height='36px')
)
out_c = widgets.Output()

def run_cases(_):
    with out_c:
        clear_output(wait=True)
        n = w_cn.value
        cases = {
            'Best (sorted)':    list(range(n)),
            'Average (random)': np.random.randint(0, 10000, size=n).tolist(),
            'Worst (reversed)': list(range(n, 0, -1)),
        }
        print(f'Best / Average / Worst Case Analysis  —  n={n}\n')
        W = 22
        hdr = f"{'Algorithm':<{W}}" + ''.join(f'  {c:>16}' for c in cases)
        print(hdr)
        print('─' * len(hdr))
        for name, fn in ALGORITHMS.items():
            row = f'{name:<{W}}'
            for data in cases.values():
                t0 = time.perf_counter(); fn(data)
                row += f'  {round((time.perf_counter()-t0)*1000, 2):>16.2f}'
            print(row)
        print('\n  (values in ms)')

w_crun.on_click(run_cases)
display(widgets.VBox([w_cn, w_crun, out_c]))


## Automated Test Cases

In [9]:
# ── AUTOMATED TEST CASES ──────────────────────────────────────────────────────
# Uses numpy to generate large random test arrays instantly.
# Verifies all 8 algorithms against np.sort (ground truth).

TEST_CASES = [
    ('Empty array',            []),
    ('Single element',         [42]),
    ('Already sorted',         list(range(10))),
    ('Reverse sorted',         list(range(10, 0, -1))),
    ('All duplicates',         [5, 5, 5, 5, 5]),
    ('Two elements',           [9, 1]),
    ('With duplicates',        [3, 1, 4, 1, 5, 9, 2, 6, 5, 3]),
    ('Random 50 elements',     np.random.randint(0, 1000, 50).tolist()),
    ('Large random n=500',     np.random.randint(0, 10000, 500).tolist()),
    ('Large random n=1000',    np.random.randint(0, 10000, 1000).tolist()),
]

w_tst = widgets.Button(
    description='▶  Run All Tests',
    button_style='info', layout=widgets.Layout(width='160px', height='36px')
)
out_tst = widgets.Output()

def run_tests(_):
    with out_tst:
        clear_output(wait=True)
        print('=== Automated Test Cases ===\n')
        total = passed = 0
        for name, fn in ALGORITHMS.items():
            print(f'  {name}')
            ok = True
            for tc_name, tc_data in TEST_CASES:
                total += 1
                try:
                    result, _, _ = fn(tc_data)
                    expected = sorted(tc_data)
                    if result == expected:
                        passed += 1; print(f'    ✓  {tc_name}')
                    else:
                        ok = False; print(f'    ✗  {tc_name}  ← WRONG OUTPUT')
                except Exception as e:
                    total -= 1; ok = False
                    print(f'    ✗  {tc_name}  ← ERROR: {e}')
            print(f'    → {"All passed ✓" if ok else "FAILURES DETECTED ✗"}')
            print()
        print(f'Total: {passed}/{total} tests passed.')

w_tst.on_click(run_tests)
display(widgets.VBox([w_tst, out_tst]))
